# Coffee Shop Demo (Google Maps / OSM)

Denne notebook viser et kort med 3 personer der bevæger sig og reagerer på regn.

**Brug:**
1. Kør alle celler fra top til bund.
2. Se personerne på kortet – de bliver røde når det regner (hver 20. sekund i 10 sekunder).
3. Tryk Ctrl+C eller stop kernel for at stoppe demoen.

# Coffee Shop Demo (Google Maps / OSM)

Denne notebook viser et kort med 3 personer der bevæger sig og reagerer på regn.

**Brug:**
1. Kør alle celler fra top til bund.
2. Se personerne på kortet – de bliver røde når det regner (hver 20. sekund i 10 sekunder).
3. Tryk Ctrl+C eller stop kernel for at stoppe demoen.

In [4]:
# Imports og konfiguration
import os
import json
import time
import random
from IPython.display import display
import importlib

import simulated_city.maplibre_live as maplibre_live
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector

importlib.reload(maplibre_live)
LiveMapLibreMap = maplibre_live.LiveMapLibreMap

cfg = load_config()
GMAP_KEY = os.environ.get('GOOGLE_MAPS_API_KEY') or getattr(cfg, 'google_maps_api_key', None)

# Vælg Google Maps tiles hvis API-nøgle findes, ellers OSM
if GMAP_KEY:
    tiles_url = f"https://mt1.google.com/vt/lyrs=m&x={'{x}'}&y={'{y}'}&z={'{z}'}&key={GMAP_KEY}"
else:
    tiles_url = 'https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png'

CITY_HALL_LNGLAT = (12.5683, 55.6761)  # København Rådhus

In [5]:
# Forbind til MQTT-broker fra config.yaml
connector = MqttConnector(cfg.mqtt, client_id_suffix='coffee-notebook')
connector.connect()
if not connector.wait_for_connection(timeout=10.0):
    raise RuntimeError('Failed to connect to MQTT broker')

print('✓ Connected to MQTT broker')
print(f'  Broker: {cfg.mqtt.host}:{cfg.mqtt.port}')

✓ Connected to MQTT broker
  Broker: 127.0.0.1:1883


In [6]:
# Start demo: 3 personer, regn hver 20. sekund
# Kør denne celle for at starte – tryk Ctrl+C eller stop kernel for at stoppe

COFFEE_SHOPS = [
    (12.5678, 55.6765),
    (12.5687, 55.6759),
    (12.5690, 55.6763),
    (12.5680, 55.6760),
]

# Opret 3 personer med tilfældige farver
persons = {}
for i in range(3):
    name = f'coffee-{i+1}'
    color = '#{:06x}'.format(random.randint(0, 0xFFFFFF))
    shop = random.choice(COFFEE_SHOPS)
    lng = shop[0] + random.uniform(-0.0005, 0.0005)
    lat = shop[1] + random.uniform(-0.0005, 0.0005)
    persons[name] = {'lng': lng, 'lat': lat, 'color': color}
    print(f'  Person created: {name} (color: {color})')

print('\\n🚀 Demo starter...')
print('Åbn map_viewer.ipynb for at se personerne på kortet\\n')

state = 'dry'
state_started = time.time()

try:
    for step in range(100):  # 100 steps = ~30 sekunder
        now = time.time()
        elapsed = now - state_started
        
        # Regn-cyklus: 20 sek tør → 10 sek regn
        if state == 'dry' and elapsed >= 20:
            state = 'rain'
            state_started = now
            connector.client.publish('weather/rain', json.dumps({'raining': True, 'timestamp': now}))
            print(f'☔ RAIN STARTS (step {step})')
        elif state == 'rain' and elapsed >= 10:
            state = 'dry'
            state_started = now
            connector.client.publish('weather/rain', json.dumps({'raining': False, 'timestamp': now}))
            print(f'☀️  RAIN STOPS (step {step})')
        
        raining = (state == 'rain')
        for name, st in persons.items():
            # Lille random walk
            st['lng'] += random.uniform(-0.00025, 0.00025)
            st['lat'] += random.uniform(-0.00025, 0.00025)
            payload = {
                'lng': float(st['lng']),
                'lat': float(st['lat']),
                'color': '#ff0000' if raining else st['color'],
                'name': name,
                'timestamp': now,
            }
            connector.client.publish(f'persons/{name}/location', json.dumps(payload))
        
        time.sleep(0.3)

except KeyboardInterrupt:
    print('\\n⏸️  Demo stopped by user')

print('\\n✓ Demo finished')
connector.disconnect()
print('✓ Disconnected from MQTT')

  Person created: coffee-1 (color: #0436b4)
  Person created: coffee-2 (color: #5d0bfa)
  Person created: coffee-3 (color: #602b77)
\n🚀 Demo starter...
Åbn map_viewer.ipynb for at se personerne på kortet\n
☔ RAIN STARTS (step 66)
☀️  RAIN STOPS (step 99)
\n✓ Demo finished


Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


✓ Disconnected from MQTT
